# Nimbus PA — acompanhamento prospectivo

Esta etapa registra dados recebidos e prepara o congelamento das previsões mensais.
O objetivo é medir desempenho em meses futuros com as versões de entrada realmente
recebidas antes da emissão. O acompanhamento inicial cobre outubro/2026–setembro/2027.

**Estado atual: coleta iniciada; nenhuma previsão real emitida.** Os testes do
congelamento usam dados sintéticos em pastas temporárias e não entram no registro real.
Ainda é necessário materializar o pacote final do híbrido com defasagens e conectar
as aquisições ERA5/SEAS5 autenticadas ao registro. O CFSv2 de setembro/2026 recebido
em 28/09 foi bloqueado por ausência dos membros esperados 21–24.

## Calendário e referência

| Informação para T | Mês utilizado |
|---|---|
| Atmosfera ERA5 final | T−4 |
| Niño 1+2, Niño 3.4, TNA e TSA | T−3 |
| SEAS5 e CFSv2 | Inicialização/emissão nominal T−1, previsão de T |
| ERSSTv5 | T−2; arquivado para pesquisa, sem entrar no modelo de referência |

O plano mantém o híbrido de duas árvores e ridge local, com pesos 0,375/0,375/0,25.
O pacote final deve ser reajustado uma única vez na janela janeiro/1993–dezembro/2022,
com as defasagens do protocolo. Essa janela fixa reaproveita o histórico oficial já
preparado e permite avaliar a degradação de um modelo sem atualização durante o ano.
Não significa que esse seja o último mês de chuva disponível em 2026.
Reajuste anual ou treino mais recente será outro experimento, registrado antes de seus resultados.

Não serão usados como pacote final os modelos dos folds históricos ou os modelos
antigos com T−1: eles têm calendários e/ou cortes diferentes. O registro recusa um
pacote incompleto, mas sua existência não substitui a auditoria do treinamento/inferência.

## O que o registro garante

- Preserva cada versão por SHA-256, sem sobrescrever os bytes de outra versão.
- Usa horário UTC da coleta real; não aceita data de recebimento retroativa.
- Armazena URL e cabeçalhos HTTP selecionados, sem cookies ou credenciais.
- Mantém `primeira_publicacao_utc=null` quando a primeira publicação é desconhecida.
- Encadeia eventos por hash e confere os arquivos antes da emissão.
- Exige todas as fontes obrigatórias, o mês correto e a entrada aprovada pela auditoria.
- Uma falha não autoriza preencher lacunas, trocar o mês ou reduzir o ensemble.
- Recusa previsão depois de iniciado o mês, outra previsão no mesmo mês, alteração de
  fontes entre inferência e congelamento, grade diferente, NaN ou precipitação negativa.
- Congela simultaneamente o híbrido e a climatologia do mesmo treinamento.
- A verificação posterior conserva observação, versão e métricas; revisões não apagam
  a primeira verificação. O painel inclui os meses sem emissão, evitando seleção de meses favoráveis.

Receber os bytes antes do mês previsto demonstra que estavam acessíveis nessa coleta;
não exige inventar a data de primeira publicação. Arquivos transformados conservam os
IDs dos originais e o código do adaptador. Isso é mais preciso que chamar `Last-Modified`
de data de publicação ou usar a inicialização do modelo como horário de chegada do arquivo.

## Limites

Este é um registro local, não um serviço com assinatura e carimbo de tempo independente.
Um administrador pode reescrever toda a cadeia ou apagar sua ponta. Para evidência externa,
poderemos ancorar o hash da previsão e da cadeia em uma versão privada hospedada, antes de T,
ou em armazenamento com retenção. A data de um commit Git definida no computador, sozinha,
não é esse comprovante. Nenhuma publicação nem ancoragem externa foi feita nesta etapa.

ERA5 final é reanálise; avaliar contra ela não equivale a medir erro em pluviômetros.
Não há garantia de entrega das fontes. Sem os arquivos corretos, o mês ficará como não emitido.
Há um plano e mecanismos de bloqueio; ainda não há um serviço de previsão em produção.

## Executar localmente

Na raiz do projeto, em Python 3.12 com `requirements.txt` desta pasta:

```bash
python pesquisa/acompanhamento/testar_registro.py
python pesquisa/acompanhamento/coletar.py --alvo 2026-10 --cfsv2
python pesquisa/acompanhamento/preparar_cfsv2.py --alvo 2026-10
```

O registro fica em `outputs/acompanhamento`, fora do Git. Para uma nova consulta ao
CFSv2, use `preparar_cfsv2.py --alvo 2026-10 --atualizar`. Cada consulta preserva o
horário verdadeiro e os bytes encontrados. Não reutiliza uma falha em cache indefinidamente.
Os comandos são pontuais; **não há agendamento automático**.

## Kaggle

Importe `Nimbus_PA_registro_prospectivo.ipynb`. A fase de coleta pública requer Internet
e nenhuma senha. Para manter a cadeia em outra sessão, anexe a saída salva anterior;
o notebook importa o registro anterior completo, mantendo seus horários. Se a pasta
estiver ausente, a nova execução começa outra cadeia e não comprova a coleta anterior.

Salvar os outputs continua sendo necessário. Reiniciar uma sessão sem guardar o registro
pode perder os únicos comprovantes locais dos downloads e das previsões.

## Integração restante para a primeira emissão

1. Adquirir ERA5 final T−4 e SEAS5 T−1 no CDS; registrar imediatamente os originais,
   validar unidades/grade/membros e preservar os recibos pais durante a transformação.
2. Reexecutar a auditoria CFSv2 quando chegar uma versão completa. Não aprovar uma exceção
   a partir de resultados do mês-alvo.
3. Ajustar e auditar o pacote final da referência no calendário definido em `plano.json`.
4. Executar a inferência somente com os IDs de `Registro.prontidao(T)`. Registrar hashes
   do modelo, código, ambiente, entradas e arquivo de saída. `Registro.congelar(...)`
   faz as verificações finais sem ler chuva observada do alvo.
5. Após T+4, obter a primeira versão ERA5 final verificada e executar a avaliação primária.
   O período de espera é uma política do projeto, não uma promessa de data do produtor.

O bloqueio atual está em `estado_inicial.json`. O arquivo é uma fotografia de 28/09/2026,
não uma consulta automática ao estado atual da fonte.

## Fontes

- [Índices mensais NOAA PSL](https://psl.noaa.gov/data/timeseries/month/); URLs exatas em `coletar.py`.
- [ERSSTv5 — NCEI](https://www.ncei.noaa.gov/pub/data/cmb/ersst/v5/netcdf/).
- [CFSv2 — IRI](https://iridl.ldeo.columbia.edu/SOURCES/.Models/.NMME/.NCEP-CFSv2/.HINDCAST/.PENTAD_SAMPLES_FULL/.prec/).
- [ERA5 — ECMWF](https://confluence.ecmwf.int/spaces/CKB/pages/76414402/ERA5%2Bdata%2Bdocumentation).
- [SEAS5/C3S — CDS](https://cds.climate.copernicus.eu/datasets/seasonal-monthly-single-levels).

O adaptador CFSv2 deriva das funções auditadas do próprio projeto; a origem e os hashes
constam em `proveniencia.json`. Nenhum código do repositório de outro competidor foi usado.


In [ ]:
from pathlib import Path
import json, sys, hashlib, runpy, shutil
BASE=Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()/'outputs'
RAIZ=BASE/'nimbus_acompanhamento'
CODIGO=RAIZ/'codigo'
REGISTRO=RAIZ/'registro'
CODIGO.mkdir(parents=True,exist_ok=True)
ARQUIVOS={'registro.py': '"""Registro local auditável. Não é assinatura nem carimbo de tempo independente."""\nfrom __future__ import annotations\n\nfrom contextlib import contextmanager\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nimport hashlib\nimport json\nimport os\nimport re\n\n\nclass ContratoError(ValueError):\n    pass\n\n\ndef exigir(ok, mensagem):\n    if not ok:\n        raise ContratoError(mensagem)\n\n\ndef agora():\n    return datetime.now(timezone.utc)\n\n\ndef instante(valor):\n    t = datetime.fromisoformat(str(valor).replace(\'Z\', \'+00:00\'))\n    exigir(t.tzinfo is not None, \'Horário sem fuso.\')\n    return t.astimezone(timezone.utc)\n\n\ndef mes(valor):\n    exigir(bool(re.fullmatch(r\'\\d{4}-\\d{2}\', str(valor))), \'Mês deve ser YYYY-MM.\')\n    t = datetime.strptime(valor, \'%Y-%m\').replace(tzinfo=timezone.utc)\n    return t\n\n\ndef deslocar(valor, n):\n    t = mes(valor)\n    k = t.year * 12 + t.month - 1 + n\n    return f\'{k // 12:04d}-{k % 12 + 1:02d}\'\n\n\ndef canonico(obj):\n    return json.dumps(obj, sort_keys=True, ensure_ascii=False, separators=(\',\', \':\'),\n                      allow_nan=False).encode(\'utf-8\')\n\n\ndef digest(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef escrever_novo(path, data):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    # Nunca sobrescrever silenciosamente um artefato registrado.\n    with path.open(\'xb\') as f:\n        f.write(data)\n        f.flush()\n        os.fsync(f.fileno())\n\n\nclass Registro:\n    def __init__(self, pasta):\n        self.pasta = Path(pasta).resolve()\n        self.pasta.mkdir(parents=True, exist_ok=True)\n        for nome in (\'objetos\', \'eventos\'):\n            (self.pasta / nome).mkdir(exist_ok=True)\n\n    @contextmanager\n    def transacao(self):\n        lock = self.pasta / \'.escrita.lock\'\n        try:\n            fd = os.open(lock, os.O_CREAT | os.O_EXCL | os.O_WRONLY)\n        except FileExistsError:\n            raise ContratoError(\'Registro em uso ou lock de execução interrompida. Não remover automaticamente.\') from None\n        os.close(fd)\n        try:\n            yield\n        finally:\n            lock.unlink()\n\n    def guardar(self, dados):\n        h = digest(dados)\n        p = self.pasta / \'objetos\' / (h + \'.bin\')\n        if p.exists():\n            exigir(p.read_bytes() == dados, \'Objeto alterado no registro.\')\n        else:\n            try:\n                escrever_novo(p, dados)\n            except FileExistsError:\n                exigir(p.read_bytes() == dados, \'Colisão de objeto.\')\n        return dict(sha256=h, bytes=len(dados))\n\n    def ler_objeto(self, ref):\n        h = ref[\'sha256\']\n        exigir(bool(re.fullmatch(\'[0-9a-f]{64}\', h)), \'Hash inválido.\')\n        p = self.pasta / \'objetos\' / (h + \'.bin\')\n        exigir(p.is_file(), \'Objeto ausente: \' + h)\n        b = p.read_bytes()\n        exigir(len(b) == ref[\'bytes\'] and digest(b) == h, \'Objeto modificado: \' + h)\n        return b\n\n    def eventos(self, conferir_objetos=False):\n        out, anterior, ultimo = [], None, None\n        for i, p in enumerate(sorted((self.pasta / \'eventos\').glob(\'*.json\')), 1):\n            e = json.loads(p.read_text(encoding=\'utf-8\'))\n            h = e.pop(\'id\')\n            exigir(digest(canonico(e)) == h, \'Evento alterado: \' + p.name)\n            exigir(p.name == f\'{i:08d}_{h}.json\', \'Sequência de eventos incompleta.\')\n            exigir(e[\'sequencia\'] == i and e[\'anterior\'] == anterior, \'Cadeia de eventos inválida.\')\n            t = instante(e[\'registrado_em_utc\'])\n            exigir(ultimo is None or t >= ultimo, \'Relógio retrocedeu; emissão bloqueada.\')\n            e[\'id\'] = h\n            if conferir_objetos:\n                for ref in e[\'dados\'].get(\'objetos\', []):\n                    self.ler_objeto(ref)\n            out.append(e)\n            anterior, ultimo = h, t\n        return out\n\n    def _adicionar(self, tipo, dados, prazo_exclusivo=None):\n        # Chamador mantém transacao; não aceita data de emissão fornecida pelo usuário.\n        existentes = self.eventos()\n        t = agora()\n        exigir(not existentes or t >= instante(existentes[-1][\'registrado_em_utc\']), \'Relógio retrocedeu.\')\n        exigir(prazo_exclusivo is None or t < prazo_exclusivo, \'Prazo ultrapassado antes de registrar a emissão.\')\n        e = dict(sequencia=len(existentes) + 1,\n                 anterior=existentes[-1][\'id\'] if existentes else None,\n                 registrado_em_utc=t.isoformat(), tipo=tipo, dados=dados)\n        e[\'id\'] = digest(canonico(e))\n        escrever_novo(self.pasta/\'eventos\'/f"{e[\'sequencia\']:08d}_{e[\'id\']}.json", canonico(e))\n        return e\n\n    def adicionar(self, tipo, dados):\n        with self.transacao():\n            return self._adicionar(tipo, dados)\n\n    def obter(self, ident, tipo=None):\n        encontrados = [e for e in self.eventos() if e[\'id\'] == ident]\n        exigir(len(encontrados) == 1, \'Evento desconhecido.\')\n        e = encontrados[0]\n        exigir(tipo is None or e[\'tipo\'] == tipo, \'Tipo de evento incorreto.\')\n        return e\n\n    def iniciar(self, plano):\n        with self.transacao():\n            anteriores = [e for e in self.eventos() if e[\'tipo\'] == \'protocolo\']\n            if anteriores:\n                exigir(len(anteriores) == 1 and anteriores[0][\'dados\'][\'plano\'] == plano,\n                       \'Plano mudou. Abrir outro registro/experimento.\')\n                return anteriores[0]\n            exigir(agora() < mes(plano[\'primeiro_mes\']), \'Protocolo não pode iniciar com mês passado/em curso.\')\n            return self._adicionar(\'protocolo\', dict(plano=plano, objetos=[]))\n\n    def protocolo(self):\n        e = [e for e in self.eventos() if e[\'tipo\'] == \'protocolo\']\n        exigir(len(e) == 1, \'Inicialize o protocolo primeiro.\')\n        return e[0]\n\n    def recibo(self, fonte, dados, metadados, transporte):\n        """Horário é o da gravação real, nunca o mtime do arquivo ou nominal do mês."""\n        exigir(fonte in self.protocolo()[\'dados\'][\'plano\'][\'fontes\'], \'Fonte fora do protocolo.\')\n        with self.transacao():\n            ref = self.guardar(dados)\n            return self._adicionar(\'fonte_recebida\', dict(fonte=fonte, objeto=ref, objetos=[ref],\n                inspecao=metadados, transporte=transporte, primeira_publicacao_utc=None,\n                evidencia=\'bytes recebidos nesta coleta; não comprova primeira publicação histórica\'))\n\n    def modelo(self, arquivos, auditoria):\n        """Registra o pacote final; não treina e não atesta sozinho o algoritmo."""\n        p = self.protocolo()\n        plano = p[\'dados\'][\'plano\']\n        exigir(set(arquivos) == set(plano[\'arquivos_modelo\']), \'Pacote de modelo incompleto ou inesperado.\')\n        exigir(auditoria[\'treino_inicio\'] == plano[\'treino_inicio\'] and\n               auditoria[\'treino_fim\'] == plano[\'treino_fim\'], \'Treinamento fora do plano.\')\n        exigir(auditoria.get(\'protocolo_id\') == p[\'id\'], \'Auditoria não corresponde ao protocolo.\')\n        with self.transacao():\n            exigir(not any(e[\'tipo\'] == \'modelo_congelado\' for e in self.eventos()), \'Modelo já congelado neste acompanhamento.\')\n            refs = {nome: self.guardar(Path(path).read_bytes()) for nome, path in arquivos.items()}\n            return self._adicionar(\'modelo_congelado\', dict(protocolo=p[\'id\'], arquivos=refs,\n                objetos=list(refs.values()), auditoria=auditoria))\n\n    def derivado(self, fonte, dados, metadados, pais, codigo):\n        exigir(bool(pais) and len(pais)==len(set(pais)), \'Originais ausentes/duplicados.\')\n        exigir(fonte in self.protocolo()[\'dados\'][\'plano\'][\'fontes\'], \'Fonte fora do protocolo.\')\n        with self.transacao():\n            for ident in pais:\n                e=self.obter(ident,\'fonte_recebida\')\n                exigir(e[\'dados\'][\'fonte\']==fonte, \'Original pertence a outra fonte.\')\n                self.ler_objeto(e[\'dados\'][\'objeto\'])\n            ref=self.guardar(dados)\n            cod=self.guardar(Path(codigo).read_bytes())\n            return self._adicionar(\'fonte_recebida\',dict(fonte=fonte,objeto=ref,objetos=[ref,cod],\n                inspecao=metadados,pais=pais,codigo=cod,primeira_publicacao_utc=None,\n                transporte=dict(metodo=\'transformacao local dos originais registrados\'),\n                evidencia=\'aquisição dos originais e processamento com código preservado\'))\n\n    def prontidao(self, alvo):\n        mes(alvo)\n        protocolo = self.protocolo()\n        p = protocolo[\'dados\'][\'plano\']\n        exigir(p[\'primeiro_mes\'] <= alvo <= p[\'ultimo_mes\'], \'Mês fora do acompanhamento.\')\n        ev = self.eventos(conferir_objetos=True)\n        fontes, faltantes = {}, []\n        for fonte, regra in p[\'fontes\'].items():\n            if not regra[\'obrigatoria\']:\n                continue\n            esperado = deslocar(alvo, -regra[\'lag\'])\n            candidatos = [e for e in ev if e[\'tipo\'] == \'fonte_recebida\'\n                and e[\'dados\'][\'fonte\'] == fonte\n                and esperado in e[\'dados\'][\'inspecao\'].get(\'meses\', [])\n                and e[\'dados\'][\'inspecao\'].get(\'validado\') is True\n                and instante(e[\'registrado_em_utc\']) < mes(alvo)]\n            if candidatos:\n                # Política fixa: última versão recebida e validada antes da emissão.\n                fontes[fonte] = candidatos[-1][\'id\']\n            else:\n                faltantes.append(f\'{fonte}: mês {esperado}\')\n        modelos = [e for e in ev if e[\'tipo\'] == \'modelo_congelado\']\n        if not modelos:\n            faltantes.append(\'modelo: pacote final do controle com defasagens ainda não registrado\')\n        if agora() >= mes(alvo):\n            faltantes.append(\'prazo: mês já começou; não retroagir emissão\')\n        return dict(mes_alvo=alvo, pronto=not faltantes, faltantes=faltantes, fontes=fontes,\n                    modelo=modelos[0][\'id\'] if modelos else None,\n                    protocolo=protocolo[\'id\'], previsao_emitida=False)\n\n    def congelar(self, alvo, arquivo, inferencia):\n        """Aceita previsão da inferência validada; nenhuma opção de backdate/overwrite."""\n        from dados import validar_previsao\n        with self.transacao():\n            exigir(not any(e[\'tipo\'] == \'previsao_congelada\' and e[\'dados\'][\'mes_alvo\'] == alvo\n                           for e in self.eventos()), \'Previsão já existe; revisão exige outra série explicitamente separada.\')\n            status = self.prontidao(alvo)\n            exigir(status[\'pronto\'], \'Emissão bloqueada: \' + \'; \'.join(status[\'faltantes\']))\n            exigir(inferencia.get(\'modelo\') == status[\'modelo\'] and\n                   inferencia.get(\'fontes\') == status[\'fontes\'] and\n                   inferencia.get(\'protocolo\') == status[\'protocolo\'], \'Inferência não usa as versões selecionadas.\')\n            b = Path(arquivo).read_bytes()\n            exigir(inferencia.get(\'previsao_sha256\') == digest(b), \'Hash da inferência diverge do arquivo.\')\n            resumo = validar_previsao(arquivo, alvo)\n            ref = self.guardar(b)\n            # Nova verificação do relógio depois da validação/gravação dos bytes.\n            exigir(agora() < mes(alvo), \'Prazo ultrapassado durante a preparação.\')\n            return self._adicionar(\'previsao_congelada\', dict(mes_alvo=alvo, protocolo=status[\'protocolo\'],\n                modelo=status[\'modelo\'], fontes=status[\'fontes\'], inferencia=inferencia,\n                previsao=ref, objetos=[ref], validacao=resumo, prazo_exclusivo_utc=mes(alvo).isoformat(),\n                carimbo_independente=False), prazo_exclusivo=mes(alvo))\n\n    def exportar_resumo(self, destino):\n        ev = self.eventos(conferir_objetos=True)\n        d = dict(eventos=len(ev), ponta=ev[-1][\'id\'] if ev else None,\n                 conferido_em_utc=agora().isoformat(),\n                 garantia=\'cadeia e hashes locais; não assinatura nem armazenamento inviolável\',\n                 fontes=[dict(id=e[\'id\'], recebido_em_utc=e[\'registrado_em_utc\'], **e[\'dados\'])\n                         for e in ev if e[\'tipo\'] == \'fonte_recebida\'],\n                 previsoes=[e for e in ev if e[\'tipo\'] == \'previsao_congelada\'])\n        Path(destino).write_text(json.dumps(d, indent=2, ensure_ascii=False), encoding=\'utf-8\')\n        return d\n', 'dados.py': '"""Inspeção dos bytes adquiridos e contrato dos campos de previsão."""\nfrom pathlib import Path\nimport tempfile\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\nfrom registro import exigir, ContratoError, mes\n\n\nLAT = np.arange(-60, 15.001, .25)\nLON = np.arange(-90, -24.999, .25)\n\n\ndef indices_psl(dados):\n    linhas = dados.decode(\'utf-8-sig\').strip().splitlines()\n    exigir(len(linhas) > 3 and len(linhas[0].split()) == 2, \'Cabeçalho PSL inválido.\')\n    a, b = [int(x) for x in linhas[0].split()]\n    exigir(1800 <= a <= b <= 2200, \'Anos declarados inválidos.\')\n    exigir(len(linhas) > b-a+2, \'Série PSL incompleta.\')\n    matriz = []\n    for ano, linha in zip(range(a, b+1), linhas[1:b-a+2]):\n        v = linha.split()\n        exigir(len(v) == 13 and int(v[0]) == ano, \'Ano/meses PSL inválidos.\')\n        matriz.extend((f\'{ano:04d}-{m:02d}\', float(x)) for m, x in enumerate(v[1:], 1))\n    sentinela = float(linhas[b-a+2].strip())\n    exigir(sentinela < -90, \'Marcador de ausentes inesperado.\')\n    pares = [(t, v) for t, v in matriz if v != sentinela]\n    exigir(pares and all(np.isfinite(v) and abs(v) < 20 for _, v in pares), \'Índice fora de escala.\')\n    return dict(validado=True, formato=\'NOAA PSL mensal\', unidades=\'degC\',\n                meses=[t for t, _ in pares], ultimo_mes=pares[-1][0],\n                ausentes=len(matriz)-len(pares), rodape=\'\\n\'.join(linhas[b-a+3:]))\n\n\ndef inspecionar_netcdf(dados, perfil, esperado=None):\n    # Bytes primeiro, validação sem alterar original; cabeçalho HTML não é NetCDF.\n    exigir(dados[:3] == b\'CDF\' or dados[:8] == b\'\\x89HDF\\r\\n\\x1a\\n\', \'Resposta não é NetCDF.\')\n    with tempfile.TemporaryDirectory() as td:\n        path = Path(td)/\'entrada.nc\'\n        path.write_bytes(dados)\n        with xr.open_dataset(path, decode_times=perfil != \'cfsv2_bruto\') as ds:\n            if perfil == \'ersstv5\':\n                exigir(ds.attrs.get(\'product_version\') == \'Version v5\', \'Versão ERSST diferente.\')\n                exigir(\'sst\' in ds and ds.sst.attrs.get(\'units\') == \'degree_C\', \'SST/unidade diferentes.\')\n                ts = pd.DatetimeIndex(ds.time.values)\n                meses = ts.strftime(\'%Y-%m\').tolist()\n                exigir(len(meses) == 1 and meses[0] == esperado, \'Mês SST não corresponde ao pedido.\')\n                exigir(ds.sizes.get(\'lat\') == 89 and ds.sizes.get(\'lon\') == 180, \'Grade ERSST inesperada.\')\n                exigir(np.array_equal(ds.lat, np.arange(-88, 90, 2)) and\n                       np.array_equal(ds.lon, np.arange(0, 360, 2)), \'Coordenadas ERSST inesperadas.\')\n                vals = ds.sst.values\n                f = vals[np.isfinite(vals)]\n                exigir(f.size > 1000 and not np.isinf(vals).any() and f.min() >= -1.801 and f.max() <= 45,\n                       \'Campo SST inválido.\')\n                return dict(validado=True, meses=meses, versao=\'ERSSTv5\', unidades=\'degree_C\',\n                    dimensoes=dict(ds.sizes), minimo=float(f.min()), maximo=float(f.max()))\n            # CFSv2 bruto é preservado mas precisa da auditoria de membros/lead/grade\n            # antes de virar entrada. Não aceitar a simples existência de NetCDF.\n            exigir(perfil == \'cfsv2_bruto\', \'Perfil de inspeção desconhecido.\')\n            return dict(validado=False, meses=[], dimensoes=dict(ds.sizes),\n                variaveis=list(ds.data_vars),\n                pendencia=\'Executar auditoria de datas reais, membros, lead 1.5, unidades e grade antes da inferência.\')\n\n\ndef validar_previsao(path, alvo):\n    with xr.open_dataset(path) as ds:\n        exigir(set(ds.data_vars) == {\'precipitacao\', \'climatologia\'}, \'Exigir previsão e climatologia de controle.\')\n        exigir(set(ds.dims) == {\'time\', \'lat\', \'lon\'} and ds.sizes[\'time\'] == 1, \'Dimensões inválidas.\')\n        exigir(np.array_equal(ds.lat, LAT) and np.array_equal(ds.lon, LON), \'Grade da previsão diferente da referência.\')\n        t = pd.DatetimeIndex(ds.time.values)\n        exigir(t.equals(pd.DatetimeIndex([mes(alvo).replace(tzinfo=None)])), \'Alvo da previsão incorreto.\')\n        for n in ds.data_vars:\n            a = ds[n]\n            exigir(a.dims == (\'time\', \'lat\', \'lon\') and a.attrs.get(\'units\') == \'mm/day\', \'Ordem/unidade de saída inválida.\')\n            exigir(np.isfinite(a.values).all() and (a.values >= 0).all(), \'Previsão contém ausentes/negativos.\')\n        return dict(pontos=int(ds.precipitacao.size), minimo=float(ds.precipitacao.min()),\n                    maximo=float(ds.precipitacao.max()), unidades=\'mm/day\')\n\n\ndef metricas(previsao, verdade):\n    """Alinhamento exato; erro de observação, grade ou unidade não é mascarado."""\n    with xr.open_dataset(previsao) as a, xr.open_dataset(verdade) as b:\n        exigir(set(b.data_vars) == {\'tp\'}, \'Observação deve conter apenas tp.\')\n        exigir(b.tp.attrs.get(\'units\') == \'mm/day\' and b.tp.dims == (\'time\',\'lat\',\'lon\'), \'Contrato da observação inválido.\')\n        exigir(np.isfinite(b.tp.values).all() and (b.tp.values >= 0).all(), \'Observação incompleta/inválida.\')\n        linhas = []\n        for nome in (\'precipitacao\', \'climatologia\'):\n            p, y = xr.align(a[nome], b.tp, join=\'exact\')\n            exigir(p.shape == y.shape, \'Observação não cobre toda a previsão.\')\n            erro = np.asarray(p.values, dtype=np.float64)-np.asarray(y.values, dtype=np.float64)\n            area = np.broadcast_to(np.cos(np.deg2rad(y.lat.values))[None,:,None], erro.shape)\n            for regiao, mask in [(\'dominio\',np.ones(len(y.lat),dtype=bool)),\n                (\'sul_35\',y.lat.values < -35),(\'35_a_15\',(y.lat.values >= -35)&(y.lat.values < -15)),\n                (\'norte_15\',y.lat.values >= -15)]:\n                e, w = erro[:,mask,:], area[:,mask,:]\n                exigir(e.size > 0, \'Região vazia.\')\n                sse, n = float((e*e).sum()), int(e.size)\n                linhas.append(dict(modelo=nome,regiao=regiao,n=n,sse=sse,\n                    soma_erro=float(e.sum()),soma_erro_absoluto=float(np.abs(e).sum()),\n                    rmse=float(np.sqrt(sse/n)),mae=float(np.abs(e).mean()),vies=float(e.mean()),\n                    sse_area=float((e*e*w).sum()),peso_area=float(w.sum()),\n                    rmse_area=float(np.sqrt((e*e*w).sum()/w.sum()))))\n        return linhas\n\n\ndef conferir_observacao_bruta(bruto, normalizado, alvo):\n    """Confere a conversão mensal ERA5; formato/expver desconhecido bloqueia."""\n    with xr.open_dataset(bruto) as ds, xr.open_dataset(normalizado) as n:\n        exigir(\'tp\' in ds and set(n.data_vars)=={\'tp\'}, \'Chuva tp ausente.\')\n        a=ds.tp\n        exigir(a.attrs.get(\'units\') in {\'m\',\'m/day\',\'m day**-1\'}, \'Unidade bruta ERA5 inesperada.\')\n        stream=a.attrs.get(\'GRIB_stream\',ds.attrs.get(\'GRIB_stream\'))\n        exigir(stream==\'moda\', \'Exigir média mensal de médias diárias, stream moda; não inferir conversão de outro produto.\')\n        if \'expver\' in a.dims:\n            exigir(a.sizes[\'expver\']==1 and int(a.expver.values[0])==1, \'ERA5T/mistura de versões não aceita.\')\n            a=a.isel(expver=0,drop=True)\n        elif \'expver\' in ds:\n            exigir(np.asarray(ds.expver).size==1 and int(np.asarray(ds.expver).ravel()[0])==1, \'Expver não final.\')\n        else:\n            ver=a.attrs.get(\'GRIB_experimentVersionNumber\',ds.attrs.get(\'GRIB_experimentVersionNumber\'))\n            exigir(str(ver) in {\'1\',\'0001\'}, \'Não há evidência de expver final no arquivo bruto.\')\n        a=a.rename({x:y for x,y in [(\'valid_time\',\'time\'),(\'latitude\',\'lat\'),(\'longitude\',\'lon\')] if x in a.dims})\n        exigir(set(a.dims)=={\'time\',\'lat\',\'lon\'} and a.sizes[\'time\']==1, \'Dimensões do alvo bruto inesperadas.\')\n        ts=pd.DatetimeIndex(a.time.values)\n        exigir(ts[0].day==1 and ts.strftime(\'%Y-%m\').tolist()==[alvo], \'Mês bruto diferente do alvo.\')\n        a=a.assign_coords(time=[np.datetime64(alvo+\'-01\')],lon=((a.lon+180)%360)-180).sortby(\'lat\').sortby(\'lon\')\n        a=a.transpose(\'time\',\'lat\',\'lon\')\n        a,b=xr.align(a,n.tp,join=\'exact\')\n        exigir(a.shape==b.shape and np.allclose(a.values*1000,b.values,rtol=1e-6,atol=1e-6),\n               \'Conversão do alvo não corresponde ao bruto ERA5 * 1000.\')\n        return dict(stream=\'moda\',expver=1,conversao=\'m * 1000 -> mm/day\',mes_alvo=alvo)\n', 'coletar.py': '"""Captura fontes públicas em versões separadas, sem publicar ou emitir previsão."""\nfrom pathlib import Path\nfrom urllib.parse import urlparse\nimport argparse\nimport json\nimport requests\nfrom registro import Registro, agora, deslocar, exigir\nfrom dados import indices_psl, inspecionar_netcdf\n\nFONTES = {\n    \'nino34\': \'https://psl.noaa.gov/data/timeseries/month/data/nino34.long.anom.data\',\n    \'nino12\': \'https://psl.noaa.gov/data/timeseries/month/data/nino12.long.anom.data\',\n    \'tna\': \'https://psl.noaa.gov/data/correlation/tna.data\',\n    \'tsa\': \'https://psl.noaa.gov/data/correlation/tsa.data\',\n}\nHOSTS = {\'psl.noaa.gov\',\'www.ncei.noaa.gov\',\'iridl.ldeo.columbia.edu\'}\n\n\ndef baixar(registro, fonte, url, perfil, esperado=None):\n    exigir(urlparse(url).scheme == \'https\' and urlparse(url).hostname in HOSTS,\n           \'Usar endpoint HTTPS autorizado do produtor.\')\n    inicio = agora().isoformat()\n    try:\n        with requests.Session() as s:\n            s.headers[\'User-Agent\'] = \'Nimbus-PA-research/1.0 (prospective-data-capture)\'\n            with s.get(url, stream=True, timeout=(20,90), allow_redirects=False) as r:\n                # Falha/redirect não grava corpo como dado válido. Não guardar cookies/tokens.\n                exigir(r.status_code == 200, f\'HTTP {r.status_code}\')\n                partes, tamanho = [], 0\n                for chunk in r.iter_content(128*1024):\n                    tamanho += len(chunk)\n                    exigir(tamanho <= 32*1024*1024, \'Resposta acima do limite de 32 MiB.\')\n                    partes.append(chunk)\n                b = b\'\'.join(partes)\n                exigir(bool(b), \'Resposta vazia.\')\n                try:\n                    meta = indices_psl(b) if perfil == \'psl\' else inspecionar_netcdf(b,perfil,esperado)\n                except Exception as erro_validacao:\n                    meta = dict(validado=False, meses=[], erro_inspecao=type(erro_validacao).__name__,\n                        pendencia=\'Resposta preservada em quarentena; não utilizar para inferência.\')\n                transporte = dict(url=url, metodo=\'HTTPS GET\', iniciado_em_utc=inicio,\n                    resposta_em_utc=agora().isoformat(), status_http=r.status_code,\n                    cabecalhos={k:r.headers[k] for k in [\'Date\',\'Last-Modified\',\'ETag\',\'Content-Type\',\'Content-Length\'] if k in r.headers},\n                    last_modified_nao_e_primeira_publicacao=True)\n        e = registro.recibo(fonte,b,meta,transporte)\n        print(f"{fonte}: recebido; validado={meta[\'validado\']}; {len(b):,} bytes; id={e[\'id\'][:12]}", flush=True)\n        return e\n    except Exception as erro:\n        # Evitar exceções HTTP que tragam tokens/cookies; URLs deste coletor são públicas.\n        registro.adicionar(\'coleta_falhou\',dict(fonte=fonte,url=url,iniciado_em_utc=inicio,\n            erro_tipo=type(erro).__name__,objetos=[]))\n        print(f\'{fonte}: coleta não aprovada ({type(erro).__name__}). Sem substituição por dados de outro mês.\',flush=True)\n        return None\n\n\ndef executar(pasta, plano_path, alvo, incluir_cfsv2=False):\n    r = Registro(pasta)\n    plano = json.loads(Path(plano_path).read_text(encoding=\'utf-8\'))\n    r.iniciar(plano)\n    r.prontidao(alvo)  # valida calendário antes do primeiro pedido\n    saidas = [baixar(r,n,u,\'psl\') for n,u in FONTES.items()]\n    t = deslocar(alvo,-2).replace(\'-\',\'\')\n    u = f\'https://www.ncei.noaa.gov/pub/data/cmb/ersst/v5/netcdf/ersst.v5.{t}.nc\'\n    saidas.append(baixar(r,\'ersstv5\',u,\'ersstv5\',deslocar(alvo,-2)))\n    if incluir_cfsv2:\n        origem = deslocar(alvo,-1)\n        ano, m = map(int,origem.split(\'-\'))\n        mon = (\'Jan\',\'Feb\',\'Mar\',\'Apr\',\'May\',\'Jun\',\'Jul\',\'Aug\',\'Sep\',\'Oct\',\'Nov\',\'Dec\')[m-1]\n        data = f\'(0000%201%20{mon}%20{ano})\'\n        raiz = \'https://iridl.ldeo.columbia.edu/SOURCES/.Models/.NMME/.NCEP-CFSv2/.HINDCAST/.PENTAD_SAMPLES_FULL/.prec/\'\n        # Arquivo dos membros, sem alterar lead nem trocar produto se a fonte falhar.\n        u = raiz+f\'S/{data}/{data}/RANGE/L/1.5/VALUES/Y/-61/16/RANGE/X/269/336/RANGE/data.nc\'\n        saidas.append(baixar(r,\'cfsv2\',u,\'cfsv2_bruto\',alvo))\n    r.exportar_resumo(Path(pasta)/\'resumo_registro.json\')\n    estado = r.prontidao(alvo)\n    (Path(pasta)/\'prontidao.json\').write_text(json.dumps(estado,indent=2,ensure_ascii=False),encoding=\'utf-8\')\n    print(json.dumps(estado,indent=2,ensure_ascii=False),flush=True)\n    return estado\n\n\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--registro\',default=\'outputs/acompanhamento\')\n    p.add_argument(\'--plano\',default=str(Path(__file__).with_name(\'plano.json\')))\n    p.add_argument(\'--alvo\',required=True)\n    p.add_argument(\'--cfsv2\',action=\'store_true\')\n    a = p.parse_args()\n    executar(a.registro,a.plano,a.alvo,a.cfsv2)\n', 'preparar_cfsv2.py': '"""Captura e audita a previsão CFSv2 do próximo mês, preservando os originais."""\nfrom pathlib import Path\nimport tempfile\nimport pandas as pd\nfrom registro import Registro, deslocar, exigir\nfrom coletar import baixar\nimport adaptador_cfsv2 as a\n\n\ndef preparar(registro, alvo, atualizar=False):\n    r=registro if isinstance(registro,Registro) else Registro(registro)\n    origem=deslocar(alvo,-1)\n    ano,mm=map(int,origem.split(\'-\'))\n    m=(\'Jan\',\'Feb\',\'Mar\',\'Apr\',\'May\',\'Jun\',\'Jul\',\'Aug\',\'Sep\',\'Oct\',\'Nov\',\'Dec\')[mm-1]\n    d=f\'(0000%201%20{m}%20{ano})\'\n    intervalo=f\'S/{d}/{d}/RANGE/\'\n    urls=[a.FONTE+intervalo+\'L/1.5/VALUES/Y/-61/16/RANGE/X/269/336/RANGE/data.nc\',\n        \'https://iridl.ldeo.columbia.edu/SOURCES/.NOAA/.NCEP/.EMC/.CFSv2/.REALTIME_ENSEMBLE/.FLXF/sampleS/\'+intervalo+\'data.nc\']\n    recibos=[]\n    for u in urls:\n        antigos=[e for e in r.eventos() if e[\'tipo\']==\'fonte_recebida\' and\n            e[\'dados\'][\'fonte\']==\'cfsv2\' and e[\'dados\'][\'transporte\'].get(\'url\')==u]\n        e=antigos[-1] if antigos and not atualizar else baixar(r,\'cfsv2\',u,\'cfsv2_bruto\',alvo)\n        exigir(e is not None,\'CFSv2 não adquirido; manter previsão bloqueada.\')\n        recibos.append(e)\n    with tempfile.TemporaryDirectory() as td:\n        paths=[]\n        for i,e in enumerate(recibos):\n            p=Path(td)/f\'original_{i}.nc\';p.write_bytes(r.ler_objeto(e[\'dados\'][\'objeto\']));paths.append(p)\n        ts=pd.DatetimeIndex([origem+\'-01\'])\n        tabela=a.tabela_inicializacoes(paths[1],ts)\n        try:\n            ds,auditoria=a.processar_bloco(paths[0],ts,tabela)\n        except ValueError as erro:\n            campos=a.carregar_membros(paths[0],ts)\n            import numpy as np\n            nvalidos=np.isfinite(campos[0]).sum(axis=(-1,-2))\n            info=dict(fonte=\'cfsv2\',mes_origem=origem,mes_alvo=alvo,\n                motivo=str(erro),pais=[e[\'id\'] for e in recibos],\n                pixels_finitos_por_membro={str(i+1):int(n) for i,n in enumerate(nvalidos)},\n                membros_esperados=(np.flatnonzero(tabela[ts[0]][\'mascara\'])+1).tolist(),\n                objetos=[],acao=\'bloquear; nenhuma exceção ou média parcial aplicada\')\n            r.adicionar(\'auditoria_falhou\',info)\n            print(\'CFSv2 bloqueado:\',info[\'motivo\'],flush=True)\n            return None\n        exigir(pd.DatetimeIndex(ds.time.values).strftime(\'%Y-%m\').tolist()==[alvo],\'Alvo CFSv2 incorreto.\')\n        p=Path(td)/\'cfsv2.nc\';ds.to_netcdf(p)\n        e=r.derivado(\'cfsv2\',p.read_bytes(),dict(validado=True,meses=[origem],meses_alvo=[alvo],\n            unidades=\'mm/day\',lead=1.5,auditoria=auditoria),[e[\'id\'] for e in recibos],Path(a.__file__))\n        print(\'CFSv2 auditado:\',origem,\'→\',alvo,\';\',auditoria[0][\'membros\'],\'membros; id\',e[\'id\'][:12],flush=True)\n        return e\n\n\nif __name__==\'__main__\':\n    import argparse\n    p=argparse.ArgumentParser();p.add_argument(\'--registro\',default=\'outputs/acompanhamento\');p.add_argument(\'--alvo\',required=True)\n    p.add_argument(\'--atualizar\',action=\'store_true\',help=\'Reconsultar a fonte e preservar nova versão dos originais.\')\n    args=p.parse_args();preparar(args.registro,args.alvo,args.atualizar)\n', 'adaptador_cfsv2.py': '"""Auditoria CFSv2 preservada do projeto: datas reais, membros, lead, grade e unidade."""\nfrom pathlib import Path\nimport hashlib\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\nimport cftime\nLEAD = 1.5\nLAT = np.arange(-61,17,dtype=np.float32)\nLON_360 = np.arange(269,337,dtype=np.float32)\nEXCECOES_MEMBROS = {}  # nenhuma excecao nova aprovada neste acompanhamento\nPOLITICA_MEMBROS = \'todos_membros_esperados_sem_lacunas_v1\'\nSCHEMA = \'nimbus_cfsv2_prospectivo_v1\'\nFONTE = \'https://iridl.ldeo.columbia.edu/SOURCES/.Models/.NMME/.NCEP-CFSv2/.HINDCAST/.PENTAD_SAMPLES_FULL/.prec/\'\n\ndef exigir(ok, mensagem):\n    if not bool(ok):\n        raise ValueError(mensagem)\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for parte in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(parte)\n    return h.hexdigest()\n\ndef meses_de_S(s):\n    exigir(str(s.attrs.get(\'units\', \'\')).strip() == \'months since 1960-01-01\', \'Unidade de S inesperada.\')\n    exigir(str(s.attrs.get(\'calendar\')) in (\'360\', \'360_day\'), \'Calendário de S inesperado.\')\n    valores = np.asarray(s.values, dtype=np.float64)\n    exigir(valores.ndim == 1 and np.isfinite(valores).all(), \'S inválido.\')\n    exigir(np.equal(valores, np.rint(valores)).all(), \'S contém mês fracionário.\')\n    return pd.DatetimeIndex([(pd.Period(\'1960-01\', freq=\'M\') + int(v)).to_timestamp() for v in valores])\n\ndef tabela_inicializacoes(path, origens):\n    with xr.open_dataset(path, decode_times=False, engine=\'netcdf4\') as d:\n        ds = d.load()\n    exigir(set(ds.data_vars) == {\'sampleS\'} and set(ds.sampleS.dims) == {\'S\', \'M\'}, \'Formato da tabela sampleS inesperado.\')\n    exigir(meses_de_S(ds.S).equals(pd.DatetimeIndex(origens)), \'Cobertura de sampleS divergente.\')\n    exigir(np.array_equal(ds.M.values, np.arange(1, 29)), \'IDs de membros da tabela inesperados.\')\n    exigir(ds.sampleS.attrs.get(\'units\') == \'days since 1960-01-01\', \'Unidade de sampleS inesperada.\')\n    exigir(str(ds.sampleS.attrs.get(\'calendar\')) in (\'365\', \'365_day\', \'noleap\'), \'Calendário sampleS inesperado.\')\n    resultado = {}\n    for k, origem in enumerate(origens):\n        numeros = ds.sampleS.transpose(\'S\', \'M\').values[k].astype(np.float64)\n        exigir(not np.isinf(numeros).any(), \'Data de inicialização infinita.\')\n        mascara = np.isfinite(numeros)\n        n = 28 if origem.month == 11 else 24\n        exigir(np.array_equal(mascara, np.arange(28) < n), f\'Tabela de membros divergente em {origem.date()}: esperados {n}.\')\n        datas_cf = cftime.num2date(numeros[mascara], \'days since 1960-01-01\', calendar=\'365_day\')\n        datas = pd.DatetimeIndex([pd.Timestamp(t.year, t.month, t.day) for t in datas_cf])\n        alvo = origem + pd.offsets.MonthBegin(1)\n        exigir((datas < alvo).all(), f\'Inicialização invade mês-alvo: {origem.date()}.\')\n        exigir(datas.min() >= origem - pd.Timedelta(days=40) and datas.max() <= origem + pd.Timedelta(days=7), \'Janela de inicialização pentadal inesperada.\')\n        exigir(len(set(datas)) == n // 4, \'Esperados quatro membros por data de inicialização.\')\n        exigir(np.all(np.unique(datas.values, return_counts=True)[1] == 4), \'Agrupamento pentadal divergente.\')\n        resultado[origem] = dict(mascara=mascara, datas=datas, numeros=numeros, fase_fonte=\'hindcast\' if origem < pd.Timestamp(\'2011-04-01\') else \'operacional\')\n    return resultado\n\ndef carregar_membros(path, origens):\n    with xr.open_dataset(path, decode_times=False, engine=\'netcdf4\') as d:\n        ds = d.load()\n    exigir(set(ds.data_vars) == {\'prec\'}, \'Arquivo deve conter somente precipitação prevista prec.\')\n    exigir(set(ds.prec.dims) == {\'S\', \'L\', \'M\', \'Y\', \'X\'}, \'Dimensões inesperadas.\')\n    exigir(meses_de_S(ds.S).equals(pd.DatetimeIndex(origens)), \'Meses do NetCDF não coincidem com o pedido.\')\n    exigir(ds.sizes[\'L\'] == 1 and float(ds.L.values[0]) == LEAD, \'Lead incorreto; exigido L=1.5.\')\n    exigir(ds.L.attrs.get(\'units\') == \'months\' and float(ds.L.attrs.get(\'pointwidth\', -1)) == 1, \'Exigida previsão mensal, não média de vários meses.\')\n    exigir(ds.prec.attrs.get(\'units\') == \'mm/day\', \'Unidade inesperada; nenhuma conversão será presumida.\')\n    exigir(ds.prec.attrs.get(\'standard_name\') in (\'precipitation_rate\', \'lwe_precipitation_rate\'), \'Variável não identificada como taxa de precipitação.\')\n    exigir(np.array_equal(ds.M.values, np.arange(1, 29)), \'IDs de membros incompletos ou duplicados.\')\n    exigir(np.array_equal(ds.Y.values, LAT) and np.array_equal(ds.X.values, LON_360), \'Grade nativa divergente do recorte de 1 grau solicitado.\')\n    exigir(ds.X.attrs.get(\'units\') == \'degree_east\' and ds.Y.attrs.get(\'units\') == \'degree_north\', \'Unidades geográficas divergentes.\')\n    valores = ds.prec.transpose(\'S\', \'L\', \'M\', \'Y\', \'X\').values[:, 0].astype(np.float64)\n    marcador = ds.prec.attrs.get(\'file_missing_value\')\n    if marcador is not None and np.isfinite(float(marcador)):\n        valores[valores == float(marcador)] = np.nan\n    return valores\n\ndef membros_utilizaveis(membros, origem, info):\n    esperados = info[\'mascara\']\n    exigir(np.isnan(membros[~esperados]).all(), f\'Membro fora da tabela com dados em {origem}.\')\n    completos = np.isfinite(membros).all(axis=(-1, -2))\n    faltantes = tuple((int(i + 1) for i in np.flatnonzero(esperados & ~completos)))\n    if faltantes:\n        exigir(faltantes == EXCECOES_MEMBROS.get(str(origem.date())), f\'Membros ausentes/parciais não previstos em {origem.date()}: {faltantes}.\')\n        exigir(np.isnan(membros[np.array(faltantes) - 1]).all(), \'A exceção exige membro inteiramente ausente; lacunas parciais ou infinitos não são aceitos.\')\n    usados = esperados & completos\n    exigir((membros[usados] >= 0).all(), f\'Precipitação negativa em {origem.date()}. Investigar.\')\n    datas = info[\'datas\'][usados[esperados]]\n    return (usados, datas, faltantes)\n\ndef processar_bloco(path, origens, tabela):\n    origens = pd.DatetimeIndex(origens)\n    membros = carregar_membros(path, origens)\n    hash_bruto = sha256(path)\n    medias, linhas, datas_min, datas_max, ns, fases = ([], [], [], [], [], [])\n    for k, origem in enumerate(origens):\n        info = tabela[origem]\n        validos, datas, faltantes = membros_utilizaveis(membros[k], origem, info)\n        v = membros[k, validos]\n        exigir(np.isfinite(v).all(), f\'Membro previsto ausente/parcial em {origem}. Não usar média parcial.\')\n        exigir((v >= 0).all(), f\'Precipitação negativa em {origem}: mínimo {v.min():.12g} mm/day. Investigar.\')\n        media = v.mean(axis=0, dtype=np.float64).astype(np.float32)\n        alvo = origem + pd.offsets.MonthBegin(1)\n        medias.append(media)\n        ns.append(int(validos.sum()))\n        datas_min.append(datas.min())\n        datas_max.append(datas.max())\n        fases.append(0 if info[\'fase_fonte\'] == \'hindcast\' else 1)\n        linhas.append(dict(time_alvo=str(alvo.date()), time_origem=str(origem.date()), lead=LEAD, membros=ns[-1], membro_ids=\',\'.join(map(str, np.flatnonzero(validos) + 1)), membros_esperados=int(info[\'mascara\'].sum()), membros_indisponiveis_ids=list(faltantes), politica_membros=POLITICA_MEMBROS, inicializacao_mais_antiga=str(datas.min().date()), inicializacao_mais_recente=str(datas.max().date()), inicializacoes_anteriores_ao_alvo=True, fase_fonte=info[\'fase_fonte\'], minimo_membros_mm_day=float(v.min()), maximo_membros_mm_day=float(v.max()), minimo_media_mm_day=float(media.min()), maximo_media_mm_day=float(media.max()), media_espacial_mm_day=float(media.mean(dtype=np.float64)), membros_ausentes_estruturais=28 - int(info[\'mascara\'].sum()), pixels_ausentes_membros_validos=0, negativos_corrigidos=0, arquivo_bruto=path.name, sha256_bruto=hash_bruto))\n    ds = xr.Dataset({\'cfsv2_tp_media\': ((\'time\', \'lat\', \'lon\'), np.stack(medias)), \'n_membros\': (\'time\', np.array(ns, dtype=np.int16)), \'time_origem\': (\'time\', origens.values), \'inicializacao_mais_antiga\': (\'time\', pd.DatetimeIndex(datas_min).values), \'inicializacao_mais_recente\': (\'time\', pd.DatetimeIndex(datas_max).values), \'fase_fonte\': (\'time\', np.array(fases, dtype=np.int8))}, coords={\'time\': (origens.to_period(\'M\') + 1).to_timestamp(), \'lat\': LAT, \'lon\': LON_360 - 360}, attrs=dict(schema=SCHEMA, fonte=FONTE, modelo=\'NCEP-CFSv2\', amostragem=\'PENTAD_SAMPLES_FULL; 24 membros, 28 em novembro\', produto=\'previsao mensal; media aritmetica local dos membros completos disponiveis\', politica_membros=POLITICA_MEMBROS, lead_iri=LEAD, referencia_temporal=\'S nominal no mes anterior ao alvo; datas reais auditadas\', observacoes_de_chuva_utilizadas=\'nenhuma\', climatologia_aplicada=\'nenhuma\', grade=\'nativa de 1 grau; interpolacao fica para o notebook de modelagem\'))\n    ds.cfsv2_tp_media.attrs = dict(units=\'mm/day\', long_name=\'Precipitacao prevista: media dos membros CFSv2\')\n    ds.lat.attrs[\'units\'], ds.lon.attrs[\'units\'] = (\'degrees_north\', \'degrees_east\')\n    ds.fase_fonte.attrs = dict(flag_values=np.array([0, 1], dtype=np.int8), flag_meanings=\'hindcast operacional\')\n    return (ds, linhas)\n', 'avaliar.py': '"""Verificação posterior das previsões congeladas; sem selecionar modelos."""\nfrom pathlib import Path\nimport json\nimport tempfile\nfrom dados import validar_previsao, metricas, conferir_observacao_bruta\nfrom registro import Registro, agora, canonico, deslocar, exigir, mes\n\n\ndef avaliar(registro, alvo, observacao, evidencia):\n    r = registro if isinstance(registro,Registro) else Registro(registro)\n    exigir(agora() >= mes(deslocar(alvo,4)), \'A verificação primária aguarda T+4 e produto final.\')\n    p = r.protocolo()\n    previsoes = [e for e in r.eventos() if e[\'tipo\']==\'previsao_congelada\' and e[\'dados\'][\'mes_alvo\']==alvo]\n    exigir(len(previsoes)==1, \'Sem previsão previamente congelada para este mês.\')\n    previsao = previsoes[0]\n    exigir(evidencia.get(\'produto\')==\'ERA5 monthly_averaged_reanalysis\' and\n           evidencia.get(\'expver\')==1 and evidencia.get(\'mes_alvo\')==alvo,\n           \'Exigir metadados do produto final e mês correto.\')\n    exigir(evidencia.get(\'url\',\'\').startswith(\'https://cds.climate.copernicus.eu/\'), \'Fonte da verificação inesperada.\')\n    raw = Path(evidencia[\'arquivo_bruto\']).read_bytes()\n    auditoria=conferir_observacao_bruta(evidencia[\'arquivo_bruto\'],observacao,alvo)\n    with r.transacao(), tempfile.TemporaryDirectory() as td:\n        fp = Path(td)/\'previsao.nc\'\n        fp.write_bytes(r.ler_objeto(previsao[\'dados\'][\'previsao\']))\n        validar_previsao(fp,alvo)\n        linhas = metricas(fp,observacao)\n        anteriores = [e for e in r.eventos() if e[\'tipo\']==\'verificacao\' and e[\'dados\'][\'mes_alvo\']==alvo]\n        ob = r.guardar(Path(observacao).read_bytes())\n        bruto = r.guardar(raw)\n        ev = {k:v for k,v in evidencia.items() if k!=\'arquivo_bruto\'}\n        ev[\'conferencia_bytes\']=auditoria\n        return r._adicionar(\'verificacao\',dict(mes_alvo=alvo,protocolo=p[\'id\'],previsao=previsao[\'id\'],\n            revisao=len(anteriores),primaria=not anteriores,metricas=linhas,evidencia=ev,\n            observacao=ob,bruto=bruto,objetos=[ob,bruto]))\n\n\ndef painel(registro):\n    r = registro if isinstance(registro,Registro) else Registro(registro)\n    p = r.protocolo()[\'dados\'][\'plano\']\n    ev = r.eventos(conferir_objetos=True)\n    atual = p[\'primeiro_mes\']\n    rows=[]\n    while atual <= p[\'ultimo_mes\']:\n        preds=[e for e in ev if e[\'tipo\']==\'previsao_congelada\' and e[\'dados\'][\'mes_alvo\']==atual]\n        obs=[e for e in ev if e[\'tipo\']==\'verificacao\' and e[\'dados\'][\'mes_alvo\']==atual and e[\'dados\'][\'primaria\']]\n        estado=(\'avaliado\' if obs else \'emitido_aguardando_observacao\' if preds else\n                \'nao_emitido_prazo_encerrado\' if agora()>=mes(atual) else \'aguardando_emissao\')\n        rows.append(dict(mes_alvo=atual,estado=estado,previsao=preds[0][\'id\'] if preds else None,\n                         verificacao=obs[0][\'id\'] if obs else None))\n        atual=deslocar(atual,1)\n    resumo=dict(meses_planejados=len(rows),meses_emitidos=sum(x[\'previsao\'] is not None for x in rows),\n        meses_avaliados=sum(x[\'verificacao\'] is not None for x in rows),meses=rows)\n    # Agregar SSE e n; não usar média simples de RMSE mensal.\n    sums={}\n    for e in ev:\n        if e[\'tipo\']!=\'verificacao\' or not e[\'dados\'][\'primaria\']: continue\n        for a in e[\'dados\'][\'metricas\']:\n            chave=a[\'modelo\']+\'/\'+a[\'regiao\']\n            z=sums.setdefault(chave,{k:0 for k in [\'n\',\'sse\',\'soma_erro\',\'soma_erro_absoluto\',\'sse_area\',\'peso_area\']})\n            for k in z: z[k]+=a[k]\n    for z in sums.values():\n        z.update(rmse=(z[\'sse\']/z[\'n\'])**.5,mae=z[\'soma_erro_absoluto\']/z[\'n\'],\n                 vies=z[\'soma_erro\']/z[\'n\'],rmse_area=(z[\'sse_area\']/z[\'peso_area\'])**.5)\n    resumo[\'metricas_primarias\']=sums\n    return resumo\n', 'testar_registro.py': '"""Testes isolados: dados sintéticos nunca entram no registro real."""\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nfrom unittest.mock import patch\nimport copy\nimport json\nimport tempfile\nimport unittest\nimport numpy as np\nimport xarray as xr\nfrom dados import LAT, LON, indices_psl\nfrom registro import Registro, ContratoError, digest, deslocar\nfrom avaliar import painel, avaliar\n\nBASE = Path(__file__).resolve().parent\nNOW = datetime(2026,9,28,20,0,tzinfo=timezone.utc)\n\n\nclass TestRegistro(unittest.TestCase):\n    def setUp(self):\n        self.temp=tempfile.TemporaryDirectory()\n        self.addCleanup(self.temp.cleanup)\n        self.raiz=Path(self.temp.name)\n        self.clock=patch(\'registro.agora\',return_value=NOW)\n        self.clock.start();self.addCleanup(self.clock.stop)\n        self.clock2=patch(\'avaliar.agora\',return_value=NOW)\n        self.clock2.start();self.addCleanup(self.clock2.stop)\n        self.r=Registro(self.raiz/\'registro\')\n        self.plano=json.loads((BASE/\'plano.json\').read_text())\n        self.protocolo=self.r.iniciar(self.plano)\n\n    def pronto(self):\n        arquivos={}\n        for nome in self.plano[\'arquivos_modelo\']:\n            p=self.raiz/nome;p.write_bytes(b\'fixture apenas sintetica\')\n            arquivos[nome]=p\n        self.r.modelo(arquivos,dict(treino_inicio=\'1993-01\',treino_fim=\'2022-12\',protocolo_id=self.protocolo[\'id\']))\n        for n,regra in self.plano[\'fontes\'].items():\n            if regra[\'obrigatoria\']:\n                self.r.recibo(n,b\'fixture sintetica \'+n.encode(),\n                    dict(validado=True,meses=[deslocar(\'2026-10\',-regra[\'lag\'])]),{\'url\':\'fixture\'})\n        self.fp=self.raiz/\'previsao.nc\'\n        a=np.ones((1,len(LAT),len(LON)),dtype=np.float32)\n        ds=xr.Dataset({n:((\'time\',\'lat\',\'lon\'),a,{\'units\':\'mm/day\'}) for n in [\'precipitacao\',\'climatologia\']},\n            coords={\'time\':[np.datetime64(\'2026-10-01\')],\'lat\':LAT,\'lon\':LON})\n        ds.to_netcdf(self.fp)\n        s=self.r.prontidao(\'2026-10\')\n        self.inf={k:s[k] for k in [\'modelo\',\'fontes\',\'protocolo\']}\n        self.inf[\'previsao_sha256\']=digest(self.fp.read_bytes())\n\n    def test_coleta_e_integridade(self):\n        e=self.r.recibo(\'nino34\',b\'abc\',dict(validado=False,meses=[]),{\'url\':\'fixture\'})\n        self.assertEqual(self.r.ler_objeto(e[\'dados\'][\'objeto\']),b\'abc\')\n        p=self.r.pasta/\'objetos\'/(e[\'dados\'][\'objeto\'][\'sha256\']+\'.bin\')\n        p.write_bytes(b\'abd\')\n        with self.assertRaises(ContratoError):self.r.eventos(conferir_objetos=True)\n\n    def test_cadeia_detecta_evento_editado(self):\n        self.r.adicionar(\'coleta_falhou\',{\'objetos\':[]})\n        p=sorted((self.r.pasta/\'eventos\').glob(\'*.json\'))[-1]\n        e=json.loads(p.read_text());e[\'tipo\']=\'fonte_recebida\'\n        p.write_text(json.dumps(e))\n        with self.assertRaises(ContratoError):self.r.eventos()\n\n    def test_protocolo_imutavel(self):\n        self.r.iniciar(self.plano)\n        outro=copy.deepcopy(self.plano);outro[\'modelo\'][\'arvores\']=900\n        with self.assertRaises(ContratoError):self.r.iniciar(outro)\n\n    def test_faltantes_bloqueiam(self):\n        s=self.r.prontidao(\'2026-10\')\n        self.assertFalse(s[\'pronto\']);self.assertEqual(len(s[\'faltantes\']),9)\n        with self.assertRaises(ContratoError):self.r.congelar(\'2026-10\',\'inexistente\',{})\n\n    def test_nao_aceita_mes_errado_ou_quarentena(self):\n        self.r.recibo(\'nino34\',b\'abc\',dict(validado=True,meses=[\'2026-08\']),{\'url\':\'fixture\'})\n        self.r.recibo(\'nino12\',b\'abc\',dict(validado=False,meses=[\'2026-07\']),{\'url\':\'fixture\'})\n        self.assertNotIn(\'nino34\',self.r.prontidao(\'2026-10\')[\'fontes\'])\n        self.assertNotIn(\'nino12\',self.r.prontidao(\'2026-10\')[\'fontes\'])\n\n    def test_congelamento_e_reemissao_bloqueada(self):\n        self.pronto()\n        e=self.r.congelar(\'2026-10\',self.fp,self.inf)\n        self.assertEqual(e[\'registrado_em_utc\'],NOW.isoformat())\n        self.assertEqual(e[\'dados\'][\'validacao\'][\'pontos\'],78561)\n        with self.assertRaises(ContratoError):self.r.congelar(\'2026-10\',self.fp,self.inf)\n\n    def test_prazo_nao_pode_ser_retroagido(self):\n        self.pronto()\n        with patch(\'registro.agora\',return_value=datetime(2026,10,1,tzinfo=timezone.utc)):\n            with self.assertRaises(ContratoError):self.r.congelar(\'2026-10\',self.fp,self.inf)\n        self.assertFalse(any(e[\'tipo\']==\'previsao_congelada\' for e in self.r.eventos()))\n\n    def test_troca_fonte_apos_inferencia_bloqueia(self):\n        self.pronto()\n        self.r.recibo(\'tna\',b\'outra versao\',dict(validado=True,meses=[\'2026-07\']),{\'url\':\'fixture\'})\n        with self.assertRaises(ContratoError):self.r.congelar(\'2026-10\',self.fp,self.inf)\n\n    def test_grid_e_valores_errados_bloqueiam(self):\n        self.pronto()\n        with xr.open_dataset(self.fp) as raw: ds=raw.load()\n        ds[\'precipitacao\'].values[0,0,0]=np.nan\n        ds.to_netcdf(self.fp,mode=\'w\')\n        self.inf[\'previsao_sha256\']=digest(self.fp.read_bytes())\n        with self.assertRaises(ContratoError):self.r.congelar(\'2026-10\',self.fp,self.inf)\n        ds[\'precipitacao\'].values[0,0,0]=1\n        ds=ds.assign_coords(lon=LON+.01);ds.to_netcdf(self.fp,mode=\'w\')\n        self.inf[\'previsao_sha256\']=digest(self.fp.read_bytes())\n        with self.assertRaises(ContratoError):self.r.congelar(\'2026-10\',self.fp,self.inf)\n\n    def test_painel_inclui_meses_sem_emissao(self):\n        d=painel(self.r)\n        self.assertEqual(d[\'meses_planejados\'],12)\n        self.assertEqual(d[\'meses_emitidos\'],0)\n        self.assertEqual(d[\'metricas_primarias\'],{})\n        with patch(\'avaliar.agora\',return_value=datetime(2026,11,2,tzinfo=timezone.utc)):\n            self.assertEqual(painel(self.r)[\'meses\'][0][\'estado\'],\'nao_emitido_prazo_encerrado\')\n\n    def test_avaliacao_nao_libera_alvo_antes_da_hora(self):\n        self.pronto();self.r.congelar(\'2026-10\',self.fp,self.inf)\n        with self.assertRaises(ContratoError):avaliar(self.r,\'2026-10\',\'inexistente\',{})\n\n    def test_avaliacao_primaria_nao_sobrescrita_por_revisao(self):\n        self.pronto();self.r.congelar(\'2026-10\',self.fp,self.inf)\n        with xr.open_dataset(self.fp) as pred: truth=pred[[\'precipitacao\']].rename({\'precipitacao\':\'tp\'}).load()\n        obs=self.raiz/\'obs.nc\';truth.to_netcdf(obs)\n        raw=truth.copy(deep=True);raw.tp.values[:]=.001\n        raw.tp.attrs=dict(units=\'m\',GRIB_stream=\'moda\',GRIB_experimentVersionNumber=\'0001\')\n        bruto=self.raiz/\'bruto.nc\';raw.to_netcdf(bruto)\n        info=dict(produto=\'ERA5 monthly_averaged_reanalysis\',expver=1,mes_alvo=\'2026-10\',\n                  url=\'https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-monthly-means\',\n                  arquivo_bruto=str(bruto))\n        mature=datetime(2027,2,2,tzinfo=timezone.utc)\n        with patch(\'registro.agora\',return_value=mature),patch(\'avaliar.agora\',return_value=mature):\n            prim=avaliar(self.r,\'2026-10\',obs,info)\n            self.assertTrue(prim[\'dados\'][\'primaria\'])\n            truth.tp.values[:]=2;truth.to_netcdf(obs,mode=\'w\')\n            with self.assertRaises(ContratoError):avaliar(self.r,\'2026-10\',obs,info)\n            raw.tp.values[:]=.002;raw.to_netcdf(bruto,mode=\'w\')\n            rev=avaliar(self.r,\'2026-10\',obs,info)\n            self.assertFalse(rev[\'dados\'][\'primaria\'])\n            self.assertEqual(painel(self.r)[\'metricas_primarias\'][\'precipitacao/dominio\'][\'rmse\'],0)\n\n    def test_derivado_exige_pais_da_mesma_fonte(self):\n        e=self.r.recibo(\'nino34\',b\'abc\',dict(validado=False,meses=[]),{\'url\':\'fixture\'})\n        codigo=self.raiz/\'adapter.py\';codigo.write_text(\'fixture\')\n        with self.assertRaises(ContratoError):\n            self.r.derivado(\'cfsv2\',b\'def\',dict(validado=True,meses=[\'2026-09\']),[e[\'id\']],codigo)\n\n    def test_parser_psl_nao_preenche_ausentes(self):\n        b=(\'2026 2026\\n2026 \'+\' \'.join([\'1\']*7+[\'-99.99\']*5)+\'\\n-99.99\\nHadISST\\n\').encode()\n        d=indices_psl(b)\n        self.assertEqual(d[\'ultimo_mes\'],\'2026-07\')\n        self.assertEqual(d[\'ausentes\'],5)\n        with self.assertRaises(ContratoError):indices_psl(b\'<html>login</html>\')\n\n\nif __name__==\'__main__\':\n    suite=unittest.defaultTestLoader.loadTestsFromTestCase(TestRegistro)\n    resultado=unittest.TextTestRunner(verbosity=2).run(suite)\n    if not resultado.wasSuccessful():raise SystemExit(1)\n', 'plano.json': '{\n  "id": "nimbus_acompanhamento_controle_defasado_v1",\n  "equipe": "Nimbus PA",\n  "primeiro_mes": "2026-10",\n  "ultimo_mes": "2027-09",\n  "prazo": "antes de 00:00 UTC do primeiro dia do mes alvo; sem emissao retroativa",\n  "treino_inicio": "1993-01",\n  "treino_fim": "2022-12",\n  "politica_modelo": "um pacote final do controle com defasagens, reajustado na janela fixa; sem SST/PCA; sem atualizar pesos ou treino durante estes doze meses",\n  "modelo": {\n    "arvores": 300,\n    "folhas": 31,\n    "taxa": 0.05,\n    "semente": 42,\n    "pontos_por_mes": 5000,\n    "ridge_alpha": 0.1,\n    "pesos": [0.375, 0.375, 0.25],\n    "escalas_residuais": [0.9, 0.875],\n    "maximo_meses_treino": 360\n  },\n  "arquivos_modelo": [\n    "arvores_seas5.txt", "arvores_multissistema.txt", "ridge.npz",\n    "clima_chuva.nc", "clima_atmosfera_indices.npz", "clima_seas5.nc",\n    "clima_cfsv2.nc", "biblioteca.py", "inferir.py", "ambiente.json",\n    "auditoria_treino.json"\n  ],\n  "fontes": {\n    "era5_sl": {"lag": 4, "obrigatoria": true, "produto": "ERA5 final mensal; tcc,t2m,sp"},\n    "era5_pl": {"lag": 4, "obrigatoria": true, "produto": "ERA5 final mensal 850 hPa; z,r,q,t,u,v"},\n    "nino34": {"lag": 3, "obrigatoria": true, "produto": "NOAA PSL nino34.long.anom.data"},\n    "nino12": {"lag": 3, "obrigatoria": true, "produto": "NOAA PSL nino12.long.anom.data"},\n    "tna": {"lag": 3, "obrigatoria": true, "produto": "NOAA PSL tna.data"},\n    "tsa": {"lag": 3, "obrigatoria": true, "produto": "NOAA PSL tsa.data"},\n    "seas5": {"lag": 1, "obrigatoria": true, "produto": "SEAS5 sistema 51, inicializacao T-1, forecastMonth 2; datas e membros auditados"},\n    "cfsv2": {"lag": 1, "obrigatoria": true, "produto": "CFSv2 IRI PENTAD_SAMPLES_FULL; emissao nominal T-1 e lead 1.5; membros auditados"},\n    "ersstv5": {"lag": 2, "obrigatoria": false, "produto": "ERSSTv5; somente arquivo de chegada, nao entra na referencia"}\n  },\n  "falha_de_fonte": "bloquear emissao; nao preencher, trocar mes, versao ou modelo silenciosamente",\n  "selecao_versao": "ultima versao recebida e validada antes da emissao; registrar IDs exatos",\n  "avaliacao": {\n    "alvo": "ERA5 mensal final total_precipitation em mm/day; nao representa pluviometro independente",\n    "primeira_verificacao": "primeira versao final conferida recebida a partir do primeiro dia de T+4",\n    "metrica_primaria": "RMSE global sqrt(soma SSE / soma n)",\n    "secundarias": ["MAE", "vies", "RMSE por area", "tres faixas de latitude", "meses emitidos/previstos"],\n    "comparacao": "climatologia do mesmo treino congelada junto a cada previsao",\n    "revisoes": "adicionar outra verificacao rotulada; nunca sobrescrever a primeira",\n    "faltantes": "reportar cobertura; nao omitir meses sem emissao dos relatorios"\n  },\n  "agendamento_automatico": false,\n  "carimbo_independente": false,\n  "limitacao": "Registro local com relogio UTC do computador e hashes. Sem assinatura/ancoragem externa; nao prova contra reescrita integral ou exclusao da ponta. Qualidade e calendario de cada adaptador devem ser auditados."\n}\n', 'README.md': '# Nimbus PA — acompanhamento prospectivo\n\nEsta etapa registra dados recebidos e prepara o congelamento das previsões mensais.\nO objetivo é medir desempenho em meses futuros com as versões de entrada realmente\nrecebidas antes da emissão. O acompanhamento inicial cobre outubro/2026–setembro/2027.\n\n**Estado atual: coleta iniciada; nenhuma previsão real emitida.** Os testes do\ncongelamento usam dados sintéticos em pastas temporárias e não entram no registro real.\nAinda é necessário materializar o pacote final do híbrido com defasagens e conectar\nas aquisições ERA5/SEAS5 autenticadas ao registro. O CFSv2 de setembro/2026 recebido\nem 28/09 foi bloqueado por ausência dos membros esperados 21–24.\n\n## Calendário e referência\n\n| Informação para T | Mês utilizado |\n|---|---|\n| Atmosfera ERA5 final | T−4 |\n| Niño 1+2, Niño 3.4, TNA e TSA | T−3 |\n| SEAS5 e CFSv2 | Inicialização/emissão nominal T−1, previsão de T |\n| ERSSTv5 | T−2; arquivado para pesquisa, sem entrar no modelo de referência |\n\nO plano mantém o híbrido de duas árvores e ridge local, com pesos 0,375/0,375/0,25.\nO pacote final deve ser reajustado uma única vez na janela janeiro/1993–dezembro/2022,\ncom as defasagens do protocolo. Essa janela fixa reaproveita o histórico oficial já\npreparado e permite avaliar a degradação de um modelo sem atualização durante o ano.\nNão significa que esse seja o último mês de chuva disponível em 2026.\nReajuste anual ou treino mais recente será outro experimento, registrado antes de seus resultados.\n\nNão serão usados como pacote final os modelos dos folds históricos ou os modelos\nantigos com T−1: eles têm calendários e/ou cortes diferentes. O registro recusa um\npacote incompleto, mas sua existência não substitui a auditoria do treinamento/inferência.\n\n## O que o registro garante\n\n- Preserva cada versão por SHA-256, sem sobrescrever os bytes de outra versão.\n- Usa horário UTC da coleta real; não aceita data de recebimento retroativa.\n- Armazena URL e cabeçalhos HTTP selecionados, sem cookies ou credenciais.\n- Mantém `primeira_publicacao_utc=null` quando a primeira publicação é desconhecida.\n- Encadeia eventos por hash e confere os arquivos antes da emissão.\n- Exige todas as fontes obrigatórias, o mês correto e a entrada aprovada pela auditoria.\n- Uma falha não autoriza preencher lacunas, trocar o mês ou reduzir o ensemble.\n- Recusa previsão depois de iniciado o mês, outra previsão no mesmo mês, alteração de\n  fontes entre inferência e congelamento, grade diferente, NaN ou precipitação negativa.\n- Congela simultaneamente o híbrido e a climatologia do mesmo treinamento.\n- A verificação posterior conserva observação, versão e métricas; revisões não apagam\n  a primeira verificação. O painel inclui os meses sem emissão, evitando seleção de meses favoráveis.\n\nReceber os bytes antes do mês previsto demonstra que estavam acessíveis nessa coleta;\nnão exige inventar a data de primeira publicação. Arquivos transformados conservam os\nIDs dos originais e o código do adaptador. Isso é mais preciso que chamar `Last-Modified`\nde data de publicação ou usar a inicialização do modelo como horário de chegada do arquivo.\n\n## Limites\n\nEste é um registro local, não um serviço com assinatura e carimbo de tempo independente.\nUm administrador pode reescrever toda a cadeia ou apagar sua ponta. Para evidência externa,\npoderemos ancorar o hash da previsão e da cadeia em uma versão privada hospedada, antes de T,\nou em armazenamento com retenção. A data de um commit Git definida no computador, sozinha,\nnão é esse comprovante. Nenhuma publicação nem ancoragem externa foi feita nesta etapa.\n\nERA5 final é reanálise; avaliar contra ela não equivale a medir erro em pluviômetros.\nNão há garantia de entrega das fontes. Sem os arquivos corretos, o mês ficará como não emitido.\nHá um plano e mecanismos de bloqueio; ainda não há um serviço de previsão em produção.\n\n## Executar localmente\n\nNa raiz do projeto, em Python 3.12 com `requirements.txt` desta pasta:\n\n```bash\npython pesquisa/acompanhamento/testar_registro.py\npython pesquisa/acompanhamento/coletar.py --alvo 2026-10 --cfsv2\npython pesquisa/acompanhamento/preparar_cfsv2.py --alvo 2026-10\n```\n\nO registro fica em `outputs/acompanhamento`, fora do Git. Para uma nova consulta ao\nCFSv2, use `preparar_cfsv2.py --alvo 2026-10 --atualizar`. Cada consulta preserva o\nhorário verdadeiro e os bytes encontrados. Não reutiliza uma falha em cache indefinidamente.\nOs comandos são pontuais; **não há agendamento automático**.\n\n## Kaggle\n\nImporte `Nimbus_PA_registro_prospectivo.ipynb`. A fase de coleta pública requer Internet\ne nenhuma senha. Para manter a cadeia em outra sessão, anexe a saída salva anterior;\no notebook importa o registro anterior completo, mantendo seus horários. Se a pasta\nestiver ausente, a nova execução começa outra cadeia e não comprova a coleta anterior.\n\nSalvar os outputs continua sendo necessário. Reiniciar uma sessão sem guardar o registro\npode perder os únicos comprovantes locais dos downloads e das previsões.\n\n## Integração restante para a primeira emissão\n\n1. Adquirir ERA5 final T−4 e SEAS5 T−1 no CDS; registrar imediatamente os originais,\n   validar unidades/grade/membros e preservar os recibos pais durante a transformação.\n2. Reexecutar a auditoria CFSv2 quando chegar uma versão completa. Não aprovar uma exceção\n   a partir de resultados do mês-alvo.\n3. Ajustar e auditar o pacote final da referência no calendário definido em `plano.json`.\n4. Executar a inferência somente com os IDs de `Registro.prontidao(T)`. Registrar hashes\n   do modelo, código, ambiente, entradas e arquivo de saída. `Registro.congelar(...)`\n   faz as verificações finais sem ler chuva observada do alvo.\n5. Após T+4, obter a primeira versão ERA5 final verificada e executar a avaliação primária.\n   O período de espera é uma política do projeto, não uma promessa de data do produtor.\n\nO bloqueio atual está em `estado_inicial.json`. O arquivo é uma fotografia de 28/09/2026,\nnão uma consulta automática ao estado atual da fonte.\n\n## Fontes\n\n- [Índices mensais NOAA PSL](https://psl.noaa.gov/data/timeseries/month/); URLs exatas em `coletar.py`.\n- [ERSSTv5 — NCEI](https://www.ncei.noaa.gov/pub/data/cmb/ersst/v5/netcdf/).\n- [CFSv2 — IRI](https://iridl.ldeo.columbia.edu/SOURCES/.Models/.NMME/.NCEP-CFSv2/.HINDCAST/.PENTAD_SAMPLES_FULL/.prec/).\n- [ERA5 — ECMWF](https://confluence.ecmwf.int/spaces/CKB/pages/76414402/ERA5%2Bdata%2Bdocumentation).\n- [SEAS5/C3S — CDS](https://cds.climate.copernicus.eu/datasets/seasonal-monthly-single-levels).\n\nO adaptador CFSv2 deriva das funções auditadas do próprio projeto; a origem e os hashes\nconstam em `proveniencia.json`. Nenhum código do repositório de outro competidor foi usado.\n', 'requirements.txt': 'numpy>=2.0,<3\npandas>=2.2,<4\nxarray>=2025.1\nrequests>=2.31,<3\nnetCDF4>=1.6,<2\ncftime>=1.6,<2\nnbformat>=5.10,<6\n'}
for nome,conteudo in ARQUIVOS.items():
    p=CODIGO/nome
    if p.exists() and p.read_text(encoding='utf-8') != conteudo:
        raise RuntimeError('Código diferente na pasta existente; preservar execução e usar outra pasta.')
    if not p.exists():p.write_text(conteudo,encoding='utf-8')
sys.path.insert(0,str(CODIGO))
from registro import Registro
# Restaurar comprovantes de uma sessão anterior. Copiar somente a cadeia e seus objetos.
if not (REGISTRO/'eventos').exists() and Path('/kaggle/input').is_dir():
    anteriores=[p.parent for p in Path('/kaggle/input').rglob('resumo_registro.json')
        if (p.parent/'eventos').is_dir() and (p.parent/'objetos').is_dir()]
    if len(anteriores)>1:raise RuntimeError('Mais de um registro anterior: anexe somente a cadeia escolhida.')
    if anteriores:
        Registro(anteriores[0]).eventos(conferir_objetos=True)
        REGISTRO.mkdir(parents=True,exist_ok=True)
        for pasta in ['eventos','objetos']:shutil.copytree(anteriores[0]/pasta,REGISTRO/pasta)
        print('Registro anterior conferido e restaurado; horários preservados.')
print('Código preparado. Ainda não foi emitida uma previsão.')


## Testes isolados
Dados sintéticos são criados somente em pastas temporárias. Nenhuma previsão de teste entra no registro real.

In [ ]:
_TESTES=runpy.run_path(str(CODIGO/'testar_registro.py'),run_name='__main__')

## Coleta pública e auditoria
Internet ligada; nenhum token. Cada resposta recebe horário real. Ausência de membros CFSv2 bloqueia a entrada. Não instalar nem migrar versões de fontes silenciosamente.

In [ ]:
from coletar import executar
from preparar_cfsv2 import preparar
ALVO='2026-10'
ESTADO=executar(REGISTRO,CODIGO/'plano.json',ALVO,incluir_cfsv2=True)
_CFS=preparar(REGISTRO,ALVO)


## Prontidão e preservação
Os arquivos de saída contêm os bytes recebidos e a cadeia de eventos. Restaurar a saída anterior é necessário para continuidade entre sessões.

In [ ]:
from avaliar import painel
from IPython.display import display, Markdown, FileLink
r=Registro(REGISTRO)
r.exportar_resumo(REGISTRO/'resumo_registro.json')
estado=r.prontidao(ALVO)
(REGISTRO/'prontidao.json').write_text(json.dumps(estado,indent=2,ensure_ascii=False),encoding='utf-8')
(REGISTRO/'painel.json').write_text(json.dumps(painel(r),indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps(estado,indent=2,ensure_ascii=False))
display(Markdown('**Salve a versão com outputs.** O registro é a evidência de chegada dos arquivos. '
    'Este notebook não reajusta o modelo, não lê chuva do alvo e não produz uma previsão enquanto houver pendências.'))
display(FileLink(str((REGISTRO/'resumo_registro.json').relative_to(BASE))))
